# AlphaZero Hex 11x11 — Entrainement Colab

Notebook pour entrainement intensif sur GPU Colab.
- Mixed precision (FP16) pour le training
- Self-play parallele (multi-processus)
- Upload/download des checkpoints
- Monitoring des metriques

## 1. Setup

In [ ]:
# Cloner le repo (adapter l'URL si necessaire)
!git clone https://github.com/votre-user/HEX_RESNET.git 2>/dev/null || echo 'Repo deja clone'
%cd HEX_RESNET/alphazero

!pip install -q torch numpy numba

In [ ]:
import os, sys, time, math
import numpy as np
import torch
import torch.nn.functional as F
from collections import deque
from concurrent.futures import ProcessPoolExecutor
from google.colab import files

# Verifier le GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} Go')
else:
    print('ATTENTION : pas de GPU detecte !')

In [ ]:
from config import *
from hex_env import HexEnv
from network import HexNet
from mcts_az import MCTSAgent
from self_play import ReplayBuffer, play_one_game, run_self_play
from evaluate import evaluate_models, evaluate_vs_random

## 2. Upload du checkpoint existant (optionnel)

Uploade `best_model.pt` et/ou `replay_buffer.npz` depuis ton serveur.

In [ ]:
os.makedirs('checkpoints', exist_ok=True)

print('Selectionne best_model.pt et/ou replay_buffer.npz :')
uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join('checkpoints', name)
    with open(path, 'wb') as f:
        f.write(data)
    print(f'  -> {path} ({len(data)/1e6:.1f} Mo)')

## 3. Configuration d'entrainement

In [ ]:
# ── Hyperparametres d'entrainement ──────────────────────────────────────────
# Ajuste selon le temps Colab disponible

ITERATIONS       = 100     # nombre d'iterations AlphaZero
GAMES_PER_IT     = 100     # parties de self-play par iteration
SIMS_SELFPLAY    = 800     # simulations MCTS par coup (self-play)
SIMS_EVAL        = 400     # simulations MCTS par coup (evaluation)
TRAIN_STEPS_IT   = 500     # pas d'entrainement par iteration
BATCH_SZ         = 512     # batch size training
LR               = 1e-3    # learning rate initial
LR_MIN           = 1e-5    # learning rate finale (cosine)
WD               = 1e-4    # weight decay
EVAL_GAMES_N     = 40      # parties d'evaluation
WIN_THRESHOLD    = 0.55    # seuil pour accepter le nouveau modele
USE_AMP          = True    # mixed precision (FP16) pour le training

# ── Self-play parallele ─────────────────────────────────────────────────────
NUM_WORKERS      = 4       # nombre de workers self-play (0 = sequentiel)

print(f'Config : {ITERATIONS} iter x {GAMES_PER_IT} parties x {SIMS_SELFPLAY} sims')
print(f'Training : {TRAIN_STEPS_IT} steps, batch={BATCH_SZ}, lr={LR}')
print(f'Self-play parallele : {NUM_WORKERS} workers')
print(f'Mixed precision : {USE_AMP}')

## 4. Fonctions d'entrainement avec mixed precision

In [ ]:
def train_step_amp(net, optimizer, scaler, states, policies, values, device):
    """Un pas d'entrainement avec mixed precision."""
    net.train()
    states_t   = torch.from_numpy(states).to(device)
    policies_t = torch.from_numpy(policies).to(device)
    values_t   = torch.from_numpy(values).unsqueeze(1).to(device)

    optimizer.zero_grad()
    with torch.amp.autocast('cuda', enabled=USE_AMP):
        log_p, v = net(states_t)
        loss_policy = -(policies_t * log_p).sum(dim=1).mean()
        loss_value  = F.mse_loss(v, values_t)
        loss = loss_policy + loss_value

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    return loss.item(), loss_value.item(), loss_policy.item()


def train_epoch_amp(net, optimizer, scheduler, scaler, buffer, steps, batch_size, device):
    """Lance `steps` pas d'entrainement avec AMP. Retourne les metriques."""
    losses, losses_v, losses_p = [], [], []
    for step in range(steps):
        if len(buffer) < batch_size:
            break
        states, policies, values = buffer.sample(batch_size)
        l, lv, lp = train_step_amp(net, optimizer, scaler, states, policies, values, device)
        losses.append(l)
        losses_v.append(lv)
        losses_p.append(lp)
        if scheduler is not None:
            scheduler.step()
    return {
        'loss':        float(np.mean(losses))   if losses   else 0.0,
        'loss_value':  float(np.mean(losses_v)) if losses_v else 0.0,
        'loss_policy': float(np.mean(losses_p)) if losses_p else 0.0,
    }

## 5. Self-play parallele

In [ ]:
def _worker_play_games(args):
    """
    Worker pour self-play parallele.
    Charge le modele depuis le disque dans chaque processus.
    Retourne les exemples d'entrainement.
    """
    model_path, num_games, sims = args
    import torch
    from network import HexNet
    from mcts_az import MCTSAgent
    from self_play import ReplayBuffer, play_one_game

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    net = HexNet().to(device)
    if os.path.isfile(model_path):
        net.load_state_dict(torch.load(model_path, map_location=device))
    net.eval()

    agent = MCTSAgent(net, device=device, sims=sims, add_dirichlet=True)
    buf = ReplayBuffer(max_size=num_games * 200)

    stats = {'blue_wins': 0, 'red_wins': 0}
    for _ in range(num_games):
        winner = play_one_game(agent, buf, augment=True)
        stats['blue_wins' if winner == 'blue' else 'red_wins'] += 1

    # Extraire les donnees du buffer
    states   = np.stack(list(buf._states))
    policies = np.stack(list(buf._policies))
    values   = np.array(list(buf._values), dtype=np.float32)
    return states, policies, values, stats


def parallel_self_play(model_path, num_games, sims, num_workers, buffer):
    """Lance le self-play en parallele sur plusieurs processus CPU."""
    if num_workers <= 0:
        # Fallback sequentiel
        net = HexNet().to(device)
        if os.path.isfile(model_path):
            net.load_state_dict(torch.load(model_path, map_location=device))
        net.eval()
        agent = MCTSAgent(net, device=device, sims=sims, add_dirichlet=True)
        stats = run_self_play(agent, buffer, num_games=num_games)
        return stats

    games_per_worker = num_games // num_workers
    remainder = num_games % num_workers
    tasks = []
    for i in range(num_workers):
        n = games_per_worker + (1 if i < remainder else 0)
        if n > 0:
            tasks.append((model_path, n, sims))

    total_stats = {'blue_wins': 0, 'red_wins': 0, 'total': num_games}
    with ProcessPoolExecutor(max_workers=num_workers) as pool:
        results = list(pool.map(_worker_play_games, tasks))

    for states, policies, values, stats in results:
        for i in range(len(states)):
            buffer.add(states[i], policies[i], float(values[i]))
        total_stats['blue_wins'] += stats['blue_wins']
        total_stats['red_wins']  += stats['red_wins']

    return total_stats

## 6. Boucle d'entrainement principale

In [ ]:
# ── Initialisation ──────────────────────────────────────────────────────────
net = HexNet().to(device)
model_path = 'checkpoints/best_model.pt'
if os.path.isfile(model_path):
    net.load_state_dict(torch.load(model_path, map_location=device))
    print(f'Checkpoint charge : {model_path}')
else:
    print('Nouveau modele initialise.')

optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=ITERATIONS * TRAIN_STEPS_IT, eta_min=LR_MIN
)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

buffer = ReplayBuffer(REPLAY_BUFFER_SIZE)
buffer_path = 'checkpoints/replay_buffer.npz'
os.makedirs('checkpoints', exist_ok=True)
n_loaded = buffer.load(buffer_path)
if n_loaded > 0:
    print(f'Buffer restaure : {n_loaded} exemples')

# Historique pour les graphes
history = {'loss': [], 'loss_value': [], 'loss_policy': [],
           'wr_vs_best': [], 'wr_vs_random': [], 'buffer_size': []}

print(f'\nPret a entrainer sur {device}')
print(f'Parametres reseau : {sum(p.numel() for p in net.parameters()):,}')

In [ ]:
# ── Boucle AlphaZero ────────────────────────────────────────────────────────
t_global = time.time()

for iteration in range(1, ITERATIONS + 1):
    print(f'\n{"="*60}')
    print(f'Iteration {iteration}/{ITERATIONS}')
    t0 = time.time()

    # ── 1. Self-play ──────────────────────────────────────────────────────
    print(f'\n[1/3] Self-play ({GAMES_PER_IT} parties, {SIMS_SELFPLAY} sims)...')
    t_sp = time.time()

    # Sauvegarder le modele courant pour les workers
    tmp_model = 'checkpoints/_current_selfplay.pt'
    torch.save(net.state_dict(), tmp_model)

    if NUM_WORKERS > 0 and device.type == 'cuda':
        stats = parallel_self_play(tmp_model, GAMES_PER_IT, SIMS_SELFPLAY, NUM_WORKERS, buffer)
    else:
        # Sequentiel (utilise le GPU directement)
        net.eval()
        agent = MCTSAgent(net, device=device, sims=SIMS_SELFPLAY, add_dirichlet=True)
        stats = run_self_play(agent, buffer, num_games=GAMES_PER_IT)

    t_sp = time.time() - t_sp
    print(f'  Blue:{stats["blue_wins"]} Red:{stats["red_wins"]} | '
          f'Buffer:{len(buffer)} | {t_sp:.0f}s')
    history['buffer_size'].append(len(buffer))

    if len(buffer) < BATCH_SZ:
        print(f'  Buffer trop petit ({len(buffer)} < {BATCH_SZ}), skip training.')
        continue

    # ── 2. Entrainement ───────────────────────────────────────────────────
    print(f'\n[2/3] Entrainement ({TRAIN_STEPS_IT} steps, batch={BATCH_SZ})...')
    t_tr = time.time()
    metrics = train_epoch_amp(net, optimizer, scheduler, scaler,
                              buffer, TRAIN_STEPS_IT, BATCH_SZ, device)
    t_tr = time.time() - t_tr
    print(f'  Loss={metrics["loss"]:.4f} val={metrics["loss_value"]:.4f} '
          f'pol={metrics["loss_policy"]:.4f} | {t_tr:.0f}s')

    history['loss'].append(metrics['loss'])
    history['loss_value'].append(metrics['loss_value'])
    history['loss_policy'].append(metrics['loss_policy'])

    # Sauvegarde intermediaire
    iter_path = f'checkpoints/model_iter_{iteration:04d}.pt'
    torch.save(net.state_dict(), iter_path)

    # ── 3. Evaluation ─────────────────────────────────────────────────────
    print(f'\n[3/3] Evaluation ({EVAL_GAMES_N} parties)...')
    t_ev = time.time()

    best_net = HexNet().to(device)
    if os.path.isfile(model_path):
        best_net.load_state_dict(torch.load(model_path, map_location=device))
        wr = evaluate_models(net, best_net, device,
                             num_games=EVAL_GAMES_N, sims=SIMS_EVAL)
    else:
        wr = 1.0  # premier modele → accepte direct

    t_ev = time.time() - t_ev
    history['wr_vs_best'].append(wr)

    if wr >= WIN_THRESHOLD:
        print(f'  Win rate: {wr:.1%} >= {WIN_THRESHOLD:.0%} -> ACCEPTE | {t_ev:.0f}s')
        torch.save(net.state_dict(), model_path)
    else:
        print(f'  Win rate: {wr:.1%} < {WIN_THRESHOLD:.0%} -> rejete | {t_ev:.0f}s')
        net.load_state_dict(torch.load(model_path, map_location=device))

    # Test vs random (toutes les 5 iterations)
    if iteration % 5 == 0:
        wr_rand = evaluate_vs_random(net, device, num_games=20, sims=200)
        history['wr_vs_random'].append(wr_rand)
        print(f'  Win rate vs random : {wr_rand:.1%}')
    else:
        history['wr_vs_random'].append(None)

    # Sauvegarde du buffer
    buffer.save(buffer_path)

    # ── Bilan iteration ───────────────────────────────────────────────────
    t_iter = time.time() - t0
    elapsed_total = time.time() - t_global
    eta = (elapsed_total / iteration) * (ITERATIONS - iteration)
    h, m = divmod(int(eta), 3600)
    m, s = divmod(m, 60)
    print(f'\n  Duree iter: {t_iter:.0f}s | Total: {elapsed_total/60:.0f}min | '
          f'ETA: {h:02d}h{m:02d}m')

print('\nEntrainement termine !')
print(f'Duree totale : {(time.time()-t_global)/3600:.1f}h')

## 7. Monitoring — Graphes d'entrainement

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss totale
ax = axes[0, 0]
ax.plot(history['loss'], label='Loss totale')
ax.set_title('Loss totale')
ax.set_xlabel('Iteration')
ax.grid(True)

# Loss decomposee
ax = axes[0, 1]
ax.plot(history['loss_value'], label='Value', color='tab:blue')
ax.plot(history['loss_policy'], label='Policy', color='tab:orange')
ax.set_title('Loss decomposee')
ax.set_xlabel('Iteration')
ax.legend()
ax.grid(True)

# Win rate vs best
ax = axes[1, 0]
ax.plot(history['wr_vs_best'], marker='o', markersize=3)
ax.axhline(y=WIN_THRESHOLD, color='r', linestyle='--', label=f'Seuil {WIN_THRESHOLD:.0%}')
ax.set_title('Win rate vs meilleur modele')
ax.set_xlabel('Iteration')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True)

# Win rate vs random
ax = axes[1, 1]
wr_r = [(i, v) for i, v in enumerate(history['wr_vs_random']) if v is not None]
if wr_r:
    ax.plot([x[0] for x in wr_r], [x[1] for x in wr_r], marker='s', color='green')
ax.set_title('Win rate vs random (toutes les 5 iter)')
ax.set_xlabel('Iteration')
ax.set_ylim(0, 1)
ax.grid(True)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=100)
plt.show()

## 8. Download du modele entraine

In [ ]:
# Telecharger le meilleur modele et le buffer
files.download('checkpoints/best_model.pt')
files.download('checkpoints/replay_buffer.npz')

## 9. Test rapide du modele

In [ ]:
# Win rate vs random
net.eval()
wr = evaluate_vs_random(net, device, num_games=50, sims=400)
print(f'Win rate vs random (50 parties, 400 sims) : {wr:.1%}')